In [ ]:
import torch
from vae.model import Encoder_model, Decoder_model
import matplotlib.pyplot as plt
import sys
import os
import json

sys.path.append('/home/pcrespo/repos/tabsyn')
from utils_train import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data_dir = '/home/pcrespo/repos/tabsyn/data/adult'
info_path = f'{data_dir}/info.json'

with open(info_path, 'r') as f:
    info = json.load(f)

X_num, X_cat, categories, d_numerical = preprocess(data_dir, task_type = info['task_type'])

X_train_num, _ = X_num
X_train_cat, _ = X_cat

X_train_num, X_test_num = X_num
X_train_cat, X_test_cat = X_cat

X_train_num, X_test_num = torch.tensor(X_train_num).float(), torch.tensor(X_test_num).float()
X_train_cat, X_test_cat =  torch.tensor(X_train_cat), torch.tensor(X_test_cat)

# Set the parameters
num_layers = 2              
d_token = 4                 
n_head = 1                  
factor = 32

pre_encoder = Encoder_model(num_layers, d_numerical, categories, d_token, n_head = n_head, factor = factor).to(device)
pre_decoder = Decoder_model(num_layers, d_numerical, categories, d_token, n_head = n_head, factor = factor).to(device)

pre_encoder.load_state_dict(torch.load('/home/pcrespo/repos/tabsyn/tabsyn/vae/ckpt/adult/encoder.pt', map_location=device))
pre_encoder.eval()
pre_decoder.load_state_dict(torch.load('/home/pcrespo/repos/tabsyn/tabsyn/vae/ckpt/adult/decoder.pt', map_location=device))
pre_decoder.eval()


In [ ]:
from tabsyn.vae.model import Model_VAE

model_path = '/home/pcrespo/repos/tabsyn/tabsyn/vae/ckpt/adult/model.pt'

model = Model_VAE(num_layers=num_layers, d_numerical=d_numerical, categories=categories, d_token=d_token, n_head=n_head, factor=factor, bias=True).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

In [34]:
X_test_num = X_test_num.to(device)
X_test_cat = X_test_cat.to(device)

with torch.no_grad():
    Recon_X_num, Recon_X_cat, mu_z, std_z = model(X_test_num, X_test_cat)

In [ ]:
Recon_X_num.shape

In [ ]:
Recon_X_cat[0]